# 使用 `send_raw_transaction` 发送交易

点击顶部栏的 `🚀` -> `Binder` 即可在线运行此示例！

本示例展示了如何在不使用 `w3.wallet` 的情况下签名交易，以及如何使用 `send_raw_transaction` 发送交易。这种方法更加灵活，但同时使用起来也更复杂。如果你只想了解如何发送交易，请参考[快速入门](./01-quickstart.ipynb)。

In [1]:
import pprint
from conflux_web3 import Web3
from conflux_web3.utils import fill_transaction_defaults

w3 = Web3(Web3.HTTPProvider("https://test.confluxrpc.com"))
account = w3.account.from_key("0x....") # replace your secret key with "0x...."

## 手动发送交易

基本上，我们遵循以下4个步骤来发送交易：

* 构建交易
* 签名交易
* 将交易发送到区块链节点
* 等待交易执行结果

如果一个账户已添加到 `w3.wallet` 中，SDK 会帮助依次完成前3个步骤，
否则你需要手动完成这些步骤。

### 构建简单交易
这里只提供必要的参数，其他参数由 `fill_transaction_defaults` 填充。
参考 [从零构建交易](./11-construct_transaction_from_scratch.ipynb) 了解如何手动填充交易的每个字段。

In [2]:
built_trivial_tx = fill_transaction_defaults(w3, {
    'from': account.address,
    'to': w3.account.create().address,
    'value': 100,
})

### 签名交易

In [3]:
signed_tx = account.sign_transaction(built_trivial_tx)

### 发送交易

In [4]:
h = w3.cfx.send_raw_transaction(signed_tx.raw_transaction)

### 等待交易执行

In [5]:
tx_receipt = h.executed()

## 与合约交互

In [6]:
# if you want to get contract object from metadata file, use
# >>> erc20_metadata = json.load(open("path/to/ERC20metadata.json"))
# >>> erc20 = web3.cfx.contract(bytecode=erc20_metadata["bytecode"], abi=erc20_metadata["abi"])
erc20 = w3.cfx.contract(name="ERC20")

# build a transaction
# parameters such as 'from'/'gasPrice'/... can be specified in `build_transaction`
built_constuct_tx = erc20.constructor(name="Coin", symbol="C", initialSupply=10**18).build_transaction({
    'from': account.address,
})
# sign the transaction
construct_tx = account.sign_transaction(built_constuct_tx)
# send & wait for execution result
contract_address = w3.cfx.send_raw_transaction(construct_tx.raw_transaction).executed()['contractCreated']
print(f"deployed contract address: {contract_address}")

deployed contract address: cfxtest:acfgtrw53v4ju9jaydmr9yyf9xyvfxpe0e8ny2fd4d


In [7]:
# interact with the deployed contract
contract = w3.cfx.contract(address=contract_address, name="ERC20")

# build a transaction
# parameters such as 'from'/'gasPrice'/... can be specified in `build_transaction`
built_transfer_tx = contract.functions.transfer(
    w3.account.create().address,
    100
).build_transaction({
    'from': account.address
})

# sign the transaction
signed_transfer_tx = account.sign_transaction(
    built_transfer_tx
)

# send & wait for execution result
print("erc20 transfer receipt: ")
pprint.pprint(
    dict(w3.cfx.send_raw_transaction(signed_transfer_tx.raw_transaction).executed())
)

erc20 transfer receipt: 
{'blockHash': HexBytes('0x9b8a0a0e5eb33d67b2c85d4e951b0251e1b972cd00f7c18c53ecacb7fd7edfd3'),
 'contractCreated': None,
 'epochNumber': 99051304,
 'from': 'cfxtest:aanhtnrex2nj56kkbws4yx0jeab34ae16pcap53w13',
 'gasCoveredBySponsor': False,
 'gasFee': 37063000000000 Drip,
 'gasUsed': 37063,
 'index': 0,
 'logs': [AttributeDict({'address': 'cfxtest:acfgtrw53v4ju9jaydmr9yyf9xyvfxpe0e8ny2fd4d', 'topics': [HexBytes('0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef'), HexBytes('0x0000000000000000000000001677ada49e168df1290c9daa4ec820039d0097e3'), HexBytes('0x0000000000000000000000001ab7c2bb12a3484451a55ed15285a4cf10196312')], 'data': HexBytes('0x0000000000000000000000000000000000000000000000000000000000000064')})],
 'logsBloom': HexBytes('0x00000020000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000008000000000000000000000000000000000000200008000000000000000000000000000000000000000000000000000000000000